<a href="https://colab.research.google.com/github/WillowsCosmic/Machine-learning-in-Python/blob/main/CGAN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CGAN

In [6]:
import tensorflow as tf
import numpy as np
import math
import matplotlib.pyplot as plt
import os

Generator Function

In [7]:
def build_generator(noise_inputs,label_inputs,image_size=28):
  #Concatenate both noise and labels
  x = tf.keras.layers.Concatenate([noise_inputs, label_inputs], axis=1)

  #Project and reshape to feed to Conv2DTranspose layer
  x = tf.keras.layers.Dense(7 * 7 * 128)(x)
  x = tf.keras.layers.Reshape((7, 7, 128))(x)

  #Use ConvTranspose
  x = tf.keras.layers.BatchNormalization()(x)
  x = tf.keras.layers.Activation('relu')(x)
  x = tf.keras.layers.Conv2DTranspose(128, kernel_size=(5,5), strides=2, padding='same')(x)

  x = tf.keras.layers.BatchNormalization()(x)
  x = tf.keras.layers.Activation('relu')(x)
  x = tf.keras.layers.Conv2DTranspose(64, kernel_size=(5,5), strides=2, padding='same')(x)

  x = tf.keras.layers.BatchNormalization()(x)

Discriminator Function

In [8]:
def build_discriminator(image_inputs, label_inputs, image_size=28):

    #Network parameters
    filter_size = 5
    num_filters = [32, 64, 128, 256]
    stride_size = [2, 2, 2, 1]

    #Build the network
    x = image_inputs

    #Make label_inputs of same size as image_inputs for concatenation
    y = tf.keras.layers.Dense(28*28)(label_inputs)
    y = tf.keras.layers.Reshape((28,28,1))(y)
    x = tf.keras.layers.Concatenate([x, y])

    x = tf.keras.layers.LeakyReLU(alpha=0.2)(x)
    x = tf.keras.layers.Conv2D(32, kernel_size=[5,5], strides=2, padding='same')(x)

    x = tf.keras.layers.LeakyReLU(alpha=0.2)(x)
    x = tf.keras.layers.Conv2D(64, kernel_size=[5,5], strides=2, padding='same')(x)

Model Building and training



In [9]:
def build_models():

    noise_size = 100
    lr = 2e-4
    decay = 6e-8

    #Build input layers
    noise_inputs = tf.keras.layers.Input(shape=(noise_size,))
    label_inputs = tf.keras.layers.Input(shape=(10,))
    image_inputs = tf.keras.layers.Input(shape=(28, 28, 1,))

    #Build Base Discriminator model
    base_discriminator = build_discriminator(image_inputs, label_inputs)

    #Define discriminator, optimizer and compile model
    discriminator = tf.keras.models.Model(inputs=[image_inputs, label_inputs],
                                        outputs=base_discriminator.outputs)
    optimizer = tf.keras.optimizers.RMSprop(lr=lr, decay=decay)
    discriminator.compile(loss='binary_crossentropy',
                         optimizer=optimizer,
                         metrics=['accuracy'])